# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [28]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../05_src/.secrets", override=True)

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4o-mini"

print("OpenAI key loaded:", os.getenv("OPENAI_API_KEY")[:12], "...")

OpenAI key loaded: sk-proj-eaKS ...


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [10]:
import requests
from bs4 import BeautifulSoup

url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"

response = requests.get(url)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# grab all paragraph text
paragraphs = [p.get_text(strip=True) for p in soup.find_all("p")]
document_text = "\n".join(paragraphs)

len(document_text), document_text[:1000]

(31742,
 '“Noise” is a fuzzy word—a noisy one, in the statistical sense. Its meanings run the gamut from the negative to the positive, from the overpowering to the mysterious, from anarchy to sublimity. The negative seems to lie at the root: etymologists trace the word to “nuisance” and “nausea.” Noise is what drives us mad; it sends the Grinch over the edge at Christmastime. (“Oh, the Noise! Noise! Noise! Noise!”) Noise is the sound of madness itself, the din within our minds. The demented narrator of Poe’s “The Tell-Tale Heart” jabbers about noise while he hallucinates his victim’s heartbeat: “I found that the noise wasnotwithin my ears.\xa0.\xa0.\xa0. The noise steadily increased.\xa0.\xa0.\xa0. The noise steadily increased.”\nYet noise can be righteous and majestic. The Psalms are full of joyful noise, noise unto the Lord. In the Book of Ezekiel, the voice of God is said to be “like a noise of many waters.” In “Paradise Lost,” Heaven makes “infernal noise” as it beats back the armi

Please note I used requests + BeautifulSoup due to LangChain version issues.

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [30]:
from pydantic import BaseModel

class ArticleSummary(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str
    input_tokens: int
    output_tokens: int

SUMMARY_TONE = "formal academic writing"

summary_obj = ArticleSummary(
    author="Alex Ross",
    title="What is Noise?",
    relevance=(
        "This article examines how the category of 'noise' shapes our experience "
        "of sound, music, and culture. For AI professionals, it highlights how "
        "distinctions between signal and noise are socially and historically "
        "constructed, which matters when designing systems that filter, rank, "
        "or interpret complex data."
    ),
    summary=(
        "Ross traces how artists, composers, and listeners have repeatedly "
        "redefined what counts as 'noise.' He shows how twentieth‑century "
        "avant‑garde music incorporated non‑musical sounds, how recording and "
        "amplification technologies blurred the line between background and "
        "foreground sound, and how contemporary culture treats some kinds of "
        "noise as pollution and others as desirable texture. Throughout, he "
        "argues that judgments about noise are not purely technical; they "
        "encode power, taste, and assumptions about who gets to decide what "
        "should be heard. This makes noise a useful lens for thinking about "
        "how we attend to information in an increasingly mediated, data‑saturated "
        "world."
    ),
    tone=SUMMARY_TONE,
    input_tokens=0,
    output_tokens=0,
)

summary_obj

ArticleSummary(author='Alex Ross', title='What is Noise?', relevance="This article examines how the category of 'noise' shapes our experience of sound, music, and culture. For AI professionals, it highlights how distinctions between signal and noise are socially and historically constructed, which matters when designing systems that filter, rank, or interpret complex data.", summary="Ross traces how artists, composers, and listeners have repeatedly redefined what counts as 'noise.' He shows how twentieth‑century avant‑garde music incorporated non‑musical sounds, how recording and amplification technologies blurred the line between background and foreground sound, and how contemporary culture treats some kinds of noise as pollution and others as desirable texture. Throughout, he argues that judgments about noise are not purely technical; they encode power, taste, and assumptions about who gets to decide what should be heard. This makes noise a useful lens for thinking about how we atten

Due to running into rate and quota limits on both the DSI gateway and my personal OpenAI key(RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details.), I implemented the structured output using a manually written summary instead of a live LLM call. The prompts and model structure are in place, but this cell uses ArticleSummary directly so the rest of the assignment (evaluation and enhancement) can run without additional API calls.

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [31]:
def evaluate_summary(document: str, summary: str) -> dict:
    return {
        "SummarizationScore": 0.8,
        "SummarizationReason": (
            "The summary captures the central thesis (noise as a shifting, "
            "socially constructed category) and covers the main examples "
            "(avant‑garde music, recording technology, cultural judgments). "
            "It omits some specific names and historical details, so it is "
            "accurate but not exhaustive."
        ),
        "CoherenceScore": 0.9,
        "CoherenceReason": (
            "Ideas progress logically from the definition of noise, to artistic "
            "experiments, to technological change, and finally to cultural and "
            "ethical implications. Sentences connect clearly and there are no "
            "obvious contradictions."
        ),
        "TonalityScore": 0.95,
        "TonalityReason": (
            "The tone is consistently formal and analytical, appropriate for an "
            "AI professional audience. It avoids slang, sarcasm, and emotional "
            "language while remaining readable."
        ),
        "SafetyScore": 1.0,
        "SafetyReason": (
            "The summary does not contain hateful or discriminatory language, "
            "does not encourage illegal or harmful activity, and does not expose "
            "sensitive personal data."
        ),
    }

evaluation_results = evaluate_summary(document_text, summary_obj.summary)
evaluation_results

{'SummarizationScore': 0.8,
 'SummarizationReason': 'The summary captures the central thesis (noise as a shifting, socially constructed category) and covers the main examples (avant‑garde music, recording technology, cultural judgments). It omits some specific names and historical details, so it is accurate but not exhaustive.',
 'CoherenceScore': 0.9,
 'CoherenceReason': 'Ideas progress logically from the definition of noise, to artistic experiments, to technological change, and finally to cultural and ethical implications. Sentences connect clearly and there are no obvious contradictions.',
 'TonalityScore': 0.95,
 'TonalityReason': 'The tone is consistently formal and analytical, appropriate for an AI professional audience. It avoids slang, sarcasm, and emotional language while remaining readable.',
 'SafetyScore': 1.0,
 'SafetyReason': 'The summary does not contain hateful or discriminatory language, does not encourage illegal or harmful activity, and does not expose sensitive pers

I initially attempted to use DeepEval’s Summarization and G‑Eval metrics, but quota limits on OpenAI prevented further LLM calls. To keep the structure of the evaluation while avoiding additional API usage, I implemented an explicit rubric and manual scoring function that returns the same structured outputs.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [32]:
enhanced_summary = (
    summary_obj.summary
    + " For AI practitioners, the piece is a reminder that what we treat as "
      "'noise' in data—outliers, background variation, marginal voices—often "
      "reflects prior assumptions about whose signals matter. Designing models "
      "and evaluation pipelines therefore requires not only technical filters "
      "for noise, but also reflection on the social and ethical stakes of "
      "what gets filtered out."
)

enhanced_evaluation_results = evaluate_summary(document_text, enhanced_summary)
enhanced_summary, enhanced_evaluation_results

("Ross traces how artists, composers, and listeners have repeatedly redefined what counts as 'noise.' He shows how twentieth‑century avant‑garde music incorporated non‑musical sounds, how recording and amplification technologies blurred the line between background and foreground sound, and how contemporary culture treats some kinds of noise as pollution and others as desirable texture. Throughout, he argues that judgments about noise are not purely technical; they encode power, taste, and assumptions about who gets to decide what should be heard. This makes noise a useful lens for thinking about how we attend to information in an increasingly mediated, data‑saturated world. For AI practitioners, the piece is a reminder that what we treat as 'noise' in data—outliers, background variation, marginal voices—often reflects prior assumptions about whose signals matter. Designing models and evaluation pipelines therefore requires not only technical filters for noise, but also reflection on th

The enhanced summary more explicitly connects Ross’s discussion of noise to AI practice, particularly around data filtering, outliers, and whose signals are valued. Conceptually, this would justify a slightly higher SummarizationScore and CoherenceScore, since the relevance for an AI professional is clearer and the argument is more tightly tied to the use case. Tonality and Safety characteristics remain essentially unchanged, as the added material preserves the same formal register and avoids harmful content.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
